In [ ]:
!pip install pillow pillow-heif matplotlib scikit-image numpy

In [ ]:
import os
import urllib.request
from PIL import Image
import pillow_heif
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Mendaftarkan format HEIF/HEIC ke library Pillow (Pencitraan Python)
pillow_heif.register_heif_opener()

print("Mengunduh gambar sampel...")
# Menggunakan gambar uji standar 'Lenna' (PNG Lossless)
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Lenna_%28test_image%29.png/512px-Lenna_%28test_image%29.png"
urllib.request.urlretrieve(url, "input.png")

# Memuat gambar asli
img_orig = Image.open("input.png").convert("RGB")

# Menyimpan sebagai JPG dan HEIC dengan setting kualitas yang sama (50)
# Kualitas sengaja disetel ke 50 agar perbedaan algoritma kompresinya terlihat
quality_setting = 50
img_orig.save("test.jpg", "JPEG", quality=quality_setting)
img_orig.save("test.heic", "HEIF", quality=quality_setting)

# 1. Mendapatkan Ukuran File
jpg_size = os.path.getsize("test.jpg") / 1024  # konversi ke KB
heic_size = os.path.getsize("test.heic") / 1024 # konversi ke KB

# Memuat kembali gambar hasil kompresi untuk dianalisis
img_jpg = Image.open("test.jpg").convert("RGB")
img_heic = Image.open("test.heic").convert("RGB")

# Mengonversi gambar ke array NumPy untuk perhitungan matematis
arr_orig = np.array(img_orig)
arr_jpg = np.array(img_jpg)
arr_heic = np.array(img_heic)

# 2. Menghitung PSNR & SSIM
# PSNR: Semakin tinggi nilainya, semakin sedikit noise/kerusakan gambar
psnr_jpg = psnr(arr_orig, arr_jpg)
psnr_heic = psnr(arr_orig, arr_heic)

# SSIM: Rentang 0 sampai 1. Semakin mendekati 1.0, struktur gambar semakin mirip aslinya
ssim_jpg = ssim(arr_orig, arr_jpg, channel_axis=2)
ssim_heic = ssim(arr_orig, arr_heic, channel_axis=2)

# --- MENCETAK HASIL ---
print(f"\n--- HASIL KOMPRESI (Tingkat Kualitas: {quality_setting}) ---")
print(f"Ukuran File : JPG = {jpg_size:.2f} KB | HEIC = {heic_size:.2f} KB")
print(f"PSNR        : JPG = {psnr_jpg:.2f} dB | HEIC = {psnr_heic:.2f} dB")
print(f"SSIM        : JPG = {ssim_jpg:.4f} | HEIC = {ssim_heic:.4f}")

# --- VISUALISASI ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(arr_orig)
axes[0].set_title("1. Original (PNG)\nBasis referensi")
axes[0].axis('off')

axes[1].imshow(arr_jpg)
axes[1].set_title(f"2. Format JPG\nUkuran: {jpg_size:.1f} KB\nSSIM: {ssim_jpg:.3f}")
axes[1].axis('off')

axes[2].imshow(arr_heic)
axes[2].set_title(f"3. Format HEIC\nUkuran: {heic_size:.1f} KB\nSSIM: {ssim_heic:.3f}")
axes[2].axis('off')

plt.tight_layout()
plt.show()